# ⚡ NeonClipping Cloud Studio (Google Colab + Google Drive)
Turn long YouTube videos, Twitch streams, or gaming VODs into viral 9:16 vertical shorts (Reels, TikTok, YouTube Shorts) with **zero Mac disk usage**.

- 🎮 **Gaming Facecam Layouts:** Face on TOP, Face on BOTTOM, or Floating in Corner.
- 🛡️ **Anti-Bot Bypass:** Android/iOS player clients & Google Drive cookies support.
- ☁️ **100% Cloud-Powered:** Heavy AI processing runs on Google's cloud GPUs.
- 💾 **Google Drive Direct Save:** Finished vertical clips save to `/content/drive/MyDrive/NeonClips`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akashrajgm/neonclipping-colab/blob/main/NeonClipping_Google_Colab.ipynb)

In [ ]:
# @title 📁 Step 1: Connect Google Drive
# @markdown Connect your Google Drive so all generated clips are saved automatically to your Drive.
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_EXPORT_DIR = "/content/drive/MyDrive/NeonClips"
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
print(f"\n✅ Google Drive mounted successfully!")
print(f"📁 Clips will be saved directly to your Drive at: {DRIVE_EXPORT_DIR}")

In [ ]:
# @title 🚀 Step 2: Setup NeonClipping Cloud Engine (with Anti-Bot Patch)
# @markdown Clones NeonClipping, installs GPU dependencies, and patches downloader to bypass YouTube datacenter bot checks.
import os
import re
import subprocess

%cd /content
if not os.path.exists("/content/neonclipping"):
    !git clone https://github.com/iamajmeena/neonclipping.git /content/neonclipping

%cd /content/neonclipping

# Install dependencies with GPU support
!pip install -q -U yt-dlp[default]
!pip install -q fastapi uvicorn python-multipart numpy Pillow opencv-python-headless streamlink pytchat faster-whisper ctranslate2

# Apply datacenter bot-bypass patch to ytdl.py
ytdl_file = "/content/neonclipping/ytdl.py"
if os.path.exists(ytdl_file):
    with open(ytdl_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    old_variants = "for clients in (['tv', 'web_safari'], ['ios'], ['mweb']):"
    new_variants = """colab_cookies = Path('/content/drive/MyDrive/NeonClips/cookies.txt')\n    if colab_cookies.is_file():\n        out.insert(0, ({**base, 'cookiefile': str(colab_cookies)}, False))\n    for clients in (['android'], ['android_creator'], ['ios'], ['web_creator'], ['tv', 'web_safari'], ['mweb']):"""
    if old_variants in content:
        content = content.replace(old_variants, new_variants)
        with open(ytdl_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print("🛡️ Applied YouTube Anti-Bot Bypass Patch to ytdl.py")

# Ensure data directory exists
os.makedirs("/content/neonclipping/data", exist_ok=True)
print("\n✅ NeonClipping Cloud Engine is ready!")

In [ ]:
# @title ✂️ Step 3 (Method A): Instant Cloud Clip Generator
# @markdown Choose YouTube URL or an MP4 file in your Google Drive (MyDrive/NeonClips).

import json
import time
import urllib.request
import urllib.error
import os
import shutil
import subprocess
import socket
import uuid

Input_Source = "YouTube_URL" # @param ["YouTube_URL", "Google_Drive_MP4"]
YouTube_URL = "https://youtu.be/BDbgF6cAGE8?si=uGD25sfXcefChjJg" # @param {type:"string"}
Drive_MP4_Filename = "video.mp4" # @param {type:"string"}

Layout = "stack" # @param ["stack", "bottom", "pip_tr", "pip_tl", "pip_br", "pip_bl", "circle", "blur"]
Streamer_Webcam_Position = "Bottom_Left" # @param ["Bottom_Left", "Top_Left", "Top_Right", "Bottom_Right", "Auto_Detect"]
Clip_Count = 5 # @param {type:"integer"}
Min_Length_Seconds = 20 # @param {type:"integer"}
Max_Length_Seconds = 60 # @param {type:"integer"}
Caption_Style = "gta" # @param ["gta", "hormozi", "mrbeast", "impact", "clean"]
Processing_Mode = "fast" # @param ["fast", "deep"]

%cd /content/neonclipping

# Restart background server if needed
subprocess.run(["pkill", "-f", "app.py"])
time.sleep(1)
print("Starting NeonClipping cloud engine...")
subprocess.Popen(["python3", "app.py"], cwd="/content/neonclipping")
time.sleep(4)

# Check for Google Drive cookies.txt
drive_cookies = "/content/drive/MyDrive/NeonClips/cookies.txt"
if os.path.exists(drive_cookies):
    print(f"🍪 Detected cookies file at: {drive_cookies} (Using for 100% bypass)")
    shutil.copy2(drive_cookies, "/content/neonclipping/data/youtube_cookies.txt")

# Map Webcam Coordinates
cam_boxes = {
    "Bottom_Left":  {'cam': {'x': 0.02, 'y': 0.50, 'w': 0.24, 'h': 0.28}, 'game': {'x': 0.24, 'y': 0.0, 'w': 0.76, 'h': 1.0}},
    "Top_Left":     {'cam': {'x': 0.00, 'y': 0.00, 'w': 0.25, 'h': 0.30}, 'game': {'x': 0.25, 'y': 0.0, 'w': 0.75, 'h': 1.0}},
    "Top_Right":    {'cam': {'x': 0.75, 'y': 0.00, 'w': 0.25, 'h': 0.30}, 'game': {'x': 0.00, 'y': 0.0, 'w': 0.75, 'h': 1.0}},
    "Bottom_Right": {'cam': {'x': 0.75, 'y': 0.70, 'w': 0.25, 'h': 0.30}, 'game': {'x': 0.00, 'y': 0.0, 'w': 0.75, 'h': 1.0}},
}

cam_config = cam_boxes.get(Streamer_Webcam_Position, {})

make_url = "http://127.0.0.1:8770/make"
payload = {
    "count": int(Clip_Count),
    "min_len": int(Min_Length_Seconds),
    "max_len": int(Max_Length_Seconds),
    "layout": Layout,
    "caption_style": Caption_Style,
    "mode": Processing_Mode,
    "language": "",
    "cam_auto": (Streamer_Webcam_Position == "Auto_Detect"),
    "top": 0.35
}

if Input_Source == "Google_Drive_MP4":
    drive_mp4 = os.path.join("/content/drive/MyDrive/NeonClips", Drive_MP4_Filename)
    if not os.path.exists(drive_mp4):
        raise FileNotFoundError(f"File {drive_mp4} not found in Google Drive!")
    local_id = uuid.uuid4().hex
    shutil.copy2(drive_mp4, f"/content/neonclipping/data/{local_id}.mp4")
    payload["id"] = local_id
    print(f"🎬 Processing Drive Video: {Drive_MP4_Filename}")
else:
    payload["url"] = YouTube_URL.strip()
    print(f"🎬 Processing YouTube: {YouTube_URL}")

if cam_config:
    payload["cam"] = cam_config["cam"]
    payload["game"] = cam_config["game"]

print(f"⚙️ Config: Layout={Layout} | Webcam={Streamer_Webcam_Position} | Count={Clip_Count} | Style={Caption_Style}")

data = json.dumps(payload).encode('utf-8')
req = urllib.request.Request(make_url, data=data, headers={'Content-Type': 'application/json'})

job_id = None
try:
    with urllib.request.urlopen(req) as resp:
        res = json.loads(resp.read().decode('utf-8'))
        job_id = res.get("job")
        print(f"🚀 Job created: {job_id}")
except urllib.error.HTTPError as e:
    err_body = e.read().decode('utf-8')
    print(f"❌ Server Error ({e.code}): {err_body}")
except Exception as e:
    print(f"❌ Request Error: {e}")

if job_id:
    print("⏳ Processing in cloud (analyzing, cutting, Whisper transcribing, rendering)...")
    status_url = f"http://127.0.0.1:8770/make/{job_id}"
    drive_dest = "/content/drive/MyDrive/NeonClips"
    os.makedirs(drive_dest, exist_ok=True)
    shown_clips = set()

    while True:
        time.sleep(3)
        try:
            with urllib.request.urlopen(status_url) as r:
                st = json.loads(r.read().decode('utf-8'))
                status = st.get("status")
                stage = st.get("stage") or status
                progress = int(round((st.get("progress") or 0) * 100))
                print(f"  [Progress: {progress}%] {stage}", end="\r")

                clips = st.get("clips", [])
                for c in clips:
                    c_file = c.get("file") if isinstance(c, dict) else c
                    if c_file and c_file not in shown_clips:
                        shown_clips.add(c_file)
                        src_clip = os.path.join("/content/neonclipping/data", f"{c_file}.mp4")
                        if not os.path.exists(src_clip):
                            src_clip = os.path.join("/content/neonclipping/data", c_file)
                        dst_clip = os.path.join(drive_dest, f"clip_{c_file}.mp4")
                        if os.path.exists(src_clip):
                            shutil.copy2(src_clip, dst_clip)
                            print(f"\n  ✅ Saved clip to Google Drive: {dst_clip}")

                if status in ("done", "complete"):
                    print(f"\n🎉 All done! Processed {len(shown_clips)} clips in {st.get('elapsed_seconds', 0)}s.")
                    print(f"📁 Check your Google Drive folder: MyDrive/NeonClips")
                    break
                elif status == "error":
                    print(f"\n❌ Job failed: {st.get('error')}")
                    break
        except Exception as e:
            time.sleep(2)

In [ ]:
# @title 🌐 Step 3 (Method B): Launch Full Web Studio in Colab
# @markdown Want to use the full interactive visual editor with facecam bounding-box dragging and transcript viewer?
# @markdown Run this cell to get a free secure public link to access the full NeonClipping Web Studio in your browser!

import subprocess
import time

%cd /content/neonclipping

# Download cloudflared for free, secure, zero-config public tunneling
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start NeonClipping backend
print("Starting NeonClipping studio server...")
subprocess.Popen(["python3", "app.py"], cwd="/content/neonclipping")
time.sleep(4)

# Start Cloudflare tunnel
print("\nTunneling public connection via Cloudflare...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8770"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    if "trycloudflare.com" in line:
        for part in line.split():
            if "trycloudflare.com" in part and part.startswith("http"):
                public_url = part
                break
        if public_url:
            break

if public_url:
    print("\n" + "="*60)
    print(f"🎉 NEONCLIPPING WEB STUDIO IS LIVE!")
    print(f"🔗 Public URL: {public_url}")
    print("="*60)
    print("Click the link above to open NeonClipping in your browser.")
    print("All processing and downloads happen on Colab (0 MB on your Mac!).")
else:
    print("Could not retrieve tunnel URL automatically. Check cloudflared output.")